# Step 1: Simulacion de una Senal SEEG Realista

## Contexto Clinico (Ejes 1-6 del Examen de Competencias)

La **estereoelectroencefalografia (SEEG)** registra la actividad electrica cerebral
mediante electrodos profundos implantados directamente en el tejido cerebral.
A diferencia del EEG de superficie, el SEEG captura **potenciales de campo local (LFP)**
con alta resolucion espacial y temporal.

### Las Tres Zonas Fundamentales

En epilepsia focal, el cerebro se puede dividir conceptualmente en tres zonas:

| Zona | Sigla | Descripcion |
|------|-------|-------------|
| **Zona Epileptogenica** | EZ | Donde se ORIGINA la crisis. Objetivo de la cirugia. |
| **Zona de Propagacion** | PZ | Regiones reclutadas DESPUES del inicio. Actividad ictal retardada. |
| **Zona No Involucrada** | NIZ | Actividad normal. No participa en la crisis. |

### Que vamos a simular?

Crearemos una senal SEEG de **12 canales** (4 electrodos x 3 contactos) que incluye:
- Un periodo **interictal** (entre crisis) con actividad de fondo
- Un periodo **ictal** (crisis) con inicio rapido en la EZ y propagacion retardada a la PZ
- **Puntas intercriticas** y **HFOs** en los canales de la EZ

### Por que simulamos en vez de usar datos reales?

Simular nos permite:
1. Controlar exactamente que hay en la senal (ground truth)
2. Verificar que nuestros metodos de analisis detectan lo que deben detectar
3. Aprender los fundamentos sin depender de datos clinicos restringidos

## 1.1 Importar Librerias

Usamos las librerias fundamentales de Python para procesamiento de senales:

In [ ]:
import numpy as np                    # Algebra lineal y operaciones con arrays
import matplotlib.pyplot as plt        # Visualizacion
from scipy.signal import butter, filtfilt, welch  # Filtros y analisis espectral

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('Librerias importadas correctamente')

## 1.2 Conceptos Fundamentales: La Senal como Combinacion Lineal

Una senal cerebral se puede modelar como una **combinacion lineal** de ondas sinusoidales:

$$s(t) = \sum_{i=1}^{N} A_i \cdot \sin(2\pi f_i t + \phi_i) + \text{ruido}$$

Donde:
- $A_i$ = **amplitud** (en microvoltios, uV) - que tan fuerte es la onda
- $f_i$ = **frecuencia** (en Hz) - que tan rapido oscila
- $\phi_i$ = **fase** (en radianes) - en que punto del ciclo empieza
- $t$ = **tiempo** (en segundos)

Cada onda sinusoidal es como un **vector base** en el espacio de senales.
La senal final es una combinacion lineal de estos vectores base.

### Bandas de frecuencia del SEEG

| Banda | Rango (Hz) | Significado clinico |
|-------|-----------|--------------------|
| Delta | 0.5 - 4 | Sueno profundo, lesion focal |
| Theta | 4 - 8 | Somnolencia, actividad hipocampal |
| Alpha | 8 - 13 | Relajacion, ritmo posterior |
| Beta | 13 - 30 | Concentracion, actividad motora |
| Gamma | 30 - 80 | Procesamiento cognitivo |
| HFO Ripples | 80 - 250 | **Biomarcador de epileptogenicidad** |
| HFO Fast Ripples | 250 - 500 | **Marcador aun mas especifico de EZ** |

## 1.3 Parametros de la Simulacion

Definimos los parametros basicos. Nota: usamos **fs=1024 Hz** (no 256 como en EEG
de superficie) porque necesitamos capturar HFOs hasta 500 Hz.

**Teorema de Nyquist**: Para capturar una frecuencia $f$, necesitamos muestrear
al menos a $2f$ Hz. Para HFOs a 500 Hz, necesitamos $fs \geq 1000$ Hz.

In [ ]:
# ============================================================
# PARAMETROS DE SIMULACION
# ============================================================

fs = 1024           # Frecuencia de muestreo (Hz)
                    # 1024 Hz permite capturar hasta 512 Hz (Nyquist)
                    # Necesario para detectar HFOs (80-500 Hz)

duration = 60       # Duracion total del registro (segundos)
                    # 60s = periodo interictal + crisis + post-ictal

n_samples = int(duration * fs)  # Total de muestras
t = np.arange(n_samples) / fs   # Vector de tiempo en segundos

# Semilla para reproducibilidad
# (misma semilla = misma senal cada vez que ejecutamos)
np.random.seed(42)

# ============================================================
# CONFIGURACION DE CANALES SEEG
# ============================================================
# Simulamos 4 electrodos profundos, cada uno con 3 contactos
# Esto es tipico en una implantacion SEEG real

channel_names = [
    # Electrodo 1: Hipocampo (EZ - Zona Epileptogenica)
    "Hip1 (EZ)",   # Contacto profundo
    "Hip2 (EZ)",   # Contacto medio
    "Hip3 (EZ)",   # Contacto superficial
    
    # Electrodo 2: Amigdala/Insula (PZ - Zona de Propagacion)
    "Amg1 (PZ)",   # Amigdala profundo
    "Amg2 (PZ)",   # Amigdala medio
    "Ins1 (PZ)",   # Insula anterior
    
    # Electrodo 3: Neocorteza Temporal (NIZ)
    "TN1 (NIZ)",   # Temporal superior
    "TN2 (NIZ)",   # Temporal medio
    "TN3 (NIZ)",   # Temporal inferior
    
    # Electrodo 4: Corteza Frontal (NIZ)
    "Fr1 (NIZ)",   # Frontal orbital
    "Fr2 (NIZ)",   # Frontal dorsolateral
    "Fr3 (NIZ)",   # Frontal medial
]

# Etiqueta de zona para cada canal
zone_labels = ["EZ"]*3 + ["PZ"]*3 + ["NIZ"]*6

n_channels = len(channel_names)

# ============================================================
# TIEMPOS DE LA CRISIS (en segundos)
# ============================================================
# La crisis dura ~20 segundos, tipico de epilepsia temporal mesial

seizure_onset_ez = 30.0   # La EZ inicia la crisis a los 30s
seizure_onset_pz = 33.0   # La PZ se involucra 3s despues (propagacion)
seizure_end = 50.0        # La crisis termina a los 50s

print(f"Configuracion de la simulacion:")
print(f"  Frecuencia de muestreo: {fs} Hz")
print(f"  Duracion: {duration} s")
print(f"  Muestras totales: {n_samples}")
print(f"  Canales: {n_channels}")
print(f"  Frecuencia maxima detectable (Nyquist): {fs/2} Hz")
print(f"")
print(f"Distribucion de zonas:")
for name, zone in zip(channel_names, zone_labels):
    print(f"  {name:15s} -> {zone}")
print(f"")
print(f"Cronologia de la crisis:")
print(f"  0-{seizure_onset_ez}s: Periodo interictal")
print(f"  {seizure_onset_ez}s: Inicio de crisis en EZ")
print(f"  {seizure_onset_pz}s: Propagacion a PZ (retraso={seizure_onset_pz-seizure_onset_ez}s)")
print(f"  {seizure_end}s: Fin de la crisis")
print(f"  {seizure_end}-{duration}s: Periodo post-ictal")

## 1.4 Funcion: Generar Actividad de Fondo

La **actividad de fondo** (background) es la senal cerebral normal que esta siempre
presente. Se compone de multiples oscilaciones en las bandas clasicas.

**Principio clave**: La senal de fondo NO es igual en todas las zonas:
- **EZ**: Tiene mayor actividad theta (4-8 Hz) y menor alpha. Esto refleja la
  disfuncion cronica del tejido epileptogenico.
- **PZ**: Actividad relativamente normal pero con leve aumento de theta.
- **NIZ**: Actividad de fondo normal con ritmos alpha y beta bien definidos.

In [ ]:
def generate_background(n_samples, fs, zone):
    """
    Genera actividad de fondo realista para un canal SEEG.
    
    La actividad de fondo se modela como una COMBINACION LINEAL de ondas
    sinusoidales en diferentes bandas de frecuencia, mas ruido.
    
    Matematicamente:
        background(t) = sum_i [ A_i * sin(2*pi*f_i*t + phi_i) ] + noise(t)
    
    Cada onda sinusoidal es un 'vector base' en el espacio de senales.
    Los coeficientes A_i determinan cuanto contribuye cada componente.
    
    Parametros
    ----------
    n_samples : int
        Numero de muestras a generar
    fs : int
        Frecuencia de muestreo en Hz
    zone : str
        'EZ', 'PZ' o 'NIZ' - determina las amplitudes relativas
    
    Retorna
    -------
    signal : numpy array
        Senal de fondo en microvoltios (uV)
    """
    t = np.arange(n_samples) / fs
    signal = np.zeros(n_samples)
    
    # --------------------------------------------------------
    # Definir amplitudes por banda segun la zona
    # --------------------------------------------------------
    # Las amplitudes reflejan la fisiopatologia:
    #   - EZ: mas theta/delta (disfuncion), menos alpha (desorganizacion)
    #   - PZ: leve aumento theta
    #   - NIZ: patron normal
    
    if zone == 'EZ':
        # La zona epileptogenica tiene actividad theta aumentada
        # y alpha reducido (refleja disfuncion neuronal cronica)
        band_config = {
            'delta':  {'freq': (0.5, 4),  'amp': 50, 'n_components': 3},
            'theta':  {'freq': (4, 8),    'amp': 45, 'n_components': 3},  # Aumentado
            'alpha':  {'freq': (8, 13),   'amp': 15, 'n_components': 2},  # Reducido
            'beta':   {'freq': (13, 30),  'amp': 10, 'n_components': 2},
            'gamma':  {'freq': (30, 80),  'amp': 5,  'n_components': 2},
        }
    elif zone == 'PZ':
        band_config = {
            'delta':  {'freq': (0.5, 4),  'amp': 40, 'n_components': 3},
            'theta':  {'freq': (4, 8),    'amp': 30, 'n_components': 2},  # Leve aumento
            'alpha':  {'freq': (8, 13),   'amp': 25, 'n_components': 2},
            'beta':   {'freq': (13, 30),  'amp': 12, 'n_components': 2},
            'gamma':  {'freq': (30, 80),  'amp': 4,  'n_components': 2},
        }
    else:  # NIZ
        band_config = {
            'delta':  {'freq': (0.5, 4),  'amp': 30, 'n_components': 2},
            'theta':  {'freq': (4, 8),    'amp': 20, 'n_components': 2},
            'alpha':  {'freq': (8, 13),   'amp': 35, 'n_components': 3},  # Alpha normal/prominente
            'beta':   {'freq': (13, 30),  'amp': 15, 'n_components': 2},
            'gamma':  {'freq': (30, 80),  'amp': 3,  'n_components': 2},
        }
    
    # --------------------------------------------------------
    # Generar cada componente sinusoidal
    # --------------------------------------------------------
    # Para cada banda, generamos N componentes con frecuencias
    # aleatorias dentro del rango de la banda.
    # Esto crea un espectro realista (no solo una frecuencia pura)
    
    for band_name, config in band_config.items():
        freq_min, freq_max = config['freq']
        base_amp = config['amp']
        n_comp = config['n_components']
        
        for _ in range(n_comp):
            # Frecuencia aleatoria dentro de la banda
            f = np.random.uniform(freq_min, freq_max)
            
            # Amplitud con variacion aleatoria (+/- 30%)
            A = base_amp * np.random.uniform(0.7, 1.3)
            
            # Fase inicial aleatoria (0 a 2*pi radianes)
            phi = np.random.uniform(0, 2 * np.pi)
            
            # La formula fundamental: A * sin(2*pi*f*t + phi)
            signal += A * np.sin(2 * np.pi * f * t + phi)
    
    # --------------------------------------------------------
    # Agregar ruido rosa (1/f)
    # --------------------------------------------------------
    # Las senales cerebrales tienen un espectro tipo '1/f':
    # la potencia disminuye con la frecuencia.
    # El ruido blanco (gaussiano) tiene potencia uniforme,
    # asi que lo combinamos con ruido rosa para ser realistas.
    
    white_noise = np.random.normal(0, 8, n_samples)
    signal += white_noise
    
    return signal

print("Funcion generate_background() definida correctamente.")
print("Esta funcion genera la actividad cerebral de fondo para cada zona.")

## 1.5 Funcion: Generar Actividad Ictal (Crisis)

La **actividad ictal** es lo que ocurre durante una crisis epileptica.
En SEEG temporal mesial, el patron tipico es:

1. **Inicio en EZ**: Actividad rapida de bajo voltaje (**LVFA** - Low Voltage Fast Activity)
   en frecuencias beta-gamma (13-80 Hz). Este patron es caracteristico del inicio ictal
   y es la base del **Indice de Epileptogenicidad (EI)** de Bartolomei et al. (2008).

2. **Propagacion a PZ**: Tras un **retraso** (delay) de varios segundos, la actividad
   ictal aparece en la zona de propagacion. El retraso es crucial para calcular el EI.

3. **Descarga ritmica**: La LVFA evoluciona a una descarga ritmica theta-alpha (4-13 Hz)
   de amplitud creciente.

4. **NIZ permanece sin cambios**: Los canales no involucrados mantienen su actividad normal.

In [ ]:
def generate_ictal_activity(n_samples, fs, zone, sz_onset_ez, sz_onset_pz, sz_end):
    """
    Genera la actividad ictal (crisis) para un canal SEEG.
    
    La actividad ictal se modela en fases:
    Fase 1 - LVFA: actividad rapida beta-gamma de bajo voltaje (inicio)
    Fase 2 - Descarga ritmica: oscilacion theta creciente (evolucion)
    
    La amplitud de la actividad ictal aumenta gradualmente usando una
    funcion 'envelope' (envolvente) que actua como modulador.
    
    Parametros
    ----------
    n_samples, fs : parametros de la senal
    zone : str - 'EZ', 'PZ' o 'NIZ'
    sz_onset_ez : float - tiempo de inicio de la crisis en EZ (segundos)
    sz_onset_pz : float - tiempo de inicio en PZ (segundos)
    sz_end : float - tiempo de fin de la crisis (segundos)
    
    Retorna
    -------
    ictal_signal : numpy array - actividad ictal en uV
    """
    t = np.arange(n_samples) / fs
    ictal_signal = np.zeros(n_samples)
    
    # NIZ no participa en la crisis
    if zone == 'NIZ':
        return ictal_signal
    
    # Determinar el tiempo de inicio segun la zona
    # Este RETRASO es fundamental para el calculo del EI
    if zone == 'EZ':
        onset = sz_onset_ez  # La EZ inicia primero
        amp_factor = 1.0     # Amplitud completa
    else:  # PZ
        onset = sz_onset_pz  # La PZ se retrasa
        amp_factor = 0.7     # Menor amplitud que la EZ
    
    # --------------------------------------------------------
    # Fase 1: LVFA (Low Voltage Fast Activity)
    # --------------------------------------------------------
    # Actividad rapida en banda beta-gamma al inicio de la crisis.
    # 'Bajo voltaje' = amplitud menor que la descarga ritmica posterior.
    # Esta fase es lo que el EI detecta: energia en frecuencias rapidas.
    
    lvfa_duration = 5.0  # La LVFA dura ~5 segundos
    lvfa_end = onset + lvfa_duration
    
    # Mascara temporal: 1 durante la LVFA, 0 fuera
    lvfa_mask = ((t >= onset) & (t < lvfa_end)).astype(float)
    
    # Suavizar los bordes con una rampa (evita artefactos)
    # La rampa sube en 0.5s al inicio y baja en 0.5s al final
    ramp_samples = int(0.5 * fs)
    onset_idx = int(onset * fs)
    lvfa_end_idx = int(lvfa_end * fs)
    
    if onset_idx + ramp_samples < n_samples:
        lvfa_mask[onset_idx:onset_idx+ramp_samples] = np.linspace(0, 1, ramp_samples)
    if lvfa_end_idx - ramp_samples >= 0 and lvfa_end_idx <= n_samples:
        ramp_end = min(ramp_samples, n_samples - (lvfa_end_idx - ramp_samples))
        lvfa_mask[lvfa_end_idx-ramp_end:lvfa_end_idx] = np.linspace(1, 0, ramp_end)
    
    # Generar la LVFA: multiples componentes beta-gamma
    lvfa = np.zeros(n_samples)
    for _ in range(5):
        f = np.random.uniform(15, 70)   # Beta a gamma
        A = np.random.uniform(20, 40)    # Amplitud moderada (low voltage)
        phi = np.random.uniform(0, 2*np.pi)
        lvfa += A * np.sin(2*np.pi*f*t + phi)
    
    ictal_signal += lvfa * lvfa_mask * amp_factor
    
    # --------------------------------------------------------
    # Fase 2: Descarga Ritmica
    # --------------------------------------------------------
    # Despues de la LVFA, la actividad evoluciona a una descarga
    # ritmica en frecuencias theta-alpha (5-10 Hz) con amplitud
    # CRECIENTE. Este patron es visible clinicamente.
    
    rhythmic_start = lvfa_end
    rhythmic_mask = ((t >= rhythmic_start) & (t < sz_end)).astype(float)
    
    # Envolvente creciente: la amplitud aumenta durante la crisis
    # Esto modela el 'reclutamiento' progresivo de neuronas
    rhythmic_env = np.zeros(n_samples)
    r_start_idx = int(rhythmic_start * fs)
    r_end_idx = min(int(sz_end * fs), n_samples)
    if r_start_idx < r_end_idx:
        r_len = r_end_idx - r_start_idx
        # Envolvente que crece linealmente hasta el 80% y luego cae
        peak_point = int(0.8 * r_len)
        rhythmic_env[r_start_idx:r_start_idx+peak_point] = np.linspace(0.3, 1.0, peak_point)
        rhythmic_env[r_start_idx+peak_point:r_end_idx] = np.linspace(1.0, 0.1, r_len - peak_point)
    
    # Generar descarga ritmica theta-alpha
    rhythmic = np.zeros(n_samples)
    for _ in range(3):
        f = np.random.uniform(5, 10)     # Theta-alpha
        A = np.random.uniform(80, 150)    # Alta amplitud
        phi = np.random.uniform(0, 2*np.pi)
        rhythmic += A * np.sin(2*np.pi*f*t + phi)
    
    ictal_signal += rhythmic * rhythmic_env * amp_factor
    
    return ictal_signal

print("Funcion generate_ictal_activity() definida correctamente.")
print("Modela dos fases: LVFA (inicio rapido) + Descarga ritmica (evolucion).")

## 1.6 Funcion: Generar Biomarcadores Interictales (Puntas y HFOs)

Durante el periodo **interictal** (entre crisis), la EZ muestra dos biomarcadores
importantes (referencia: Eje 3 del examen - Chavez Q1, Q5):

### Puntas Intercriticas (Interictal Spikes)
- Son ondas agudas de alta amplitud que duran 20-70 ms
- Ocurren irregularmente (0.5-3 por segundo en EZ)
- Se modelan como ondas gaussianas moduladas

### Oscilaciones de Alta Frecuencia (HFOs)
- **Ripples**: 80-250 Hz, duracion 50-200 ms
- **Fast Ripples**: 250-500 Hz, duracion 10-50 ms
- Son el biomarcador mas especifico de la EZ (Roehri & Bartolomei, 2019)

### Por que solo en la EZ?
Los HFOs y las puntas reflejan la **hiperexcitabilidad** local del tejido
epileptogenico. Son generados por poblaciones neuronales patologicamente
sincronizadas. Esta es la razon por la que se usan como marcadores de la EZ.

In [ ]:
def generate_interictal_biomarkers(n_samples, fs, zone, sz_onset):
    """
    Genera puntas intercriticas y HFOs para canales de la EZ.
    
    Solo se generan durante el periodo INTERICTAL (antes de la crisis)
    y solo en la EZ (donde el tejido es hiperexcitable).
    
    Modelo matematico de una punta:
        spike(t) = A * exp(-(t-t0)^2 / (2*sigma^2))
    Esto es una funcion Gaussiana centrada en t0 con ancho sigma.
    
    Modelo de un HFO:
        hfo(t) = A * sin(2*pi*f*t) * exp(-(t-t0)^2 / (2*sigma^2))
    Una sinusoide de alta frecuencia modulada por una Gaussiana (burst corto).
    """
    t = np.arange(n_samples) / fs
    biomarker_signal = np.zeros(n_samples)
    
    # Solo la EZ genera estos biomarcadores
    if zone != 'EZ':
        return biomarker_signal
    
    # --------------------------------------------------------
    # Puntas intercriticas
    # --------------------------------------------------------
    # Generamos puntas aleatorias SOLO en el periodo interictal
    # (antes de la crisis, t < sz_onset)
    
    spike_rate = 1.5  # puntas por segundo (promedio)
    interictal_duration = sz_onset  # segundos de periodo interictal
    n_spikes = int(spike_rate * interictal_duration)
    
    # Tiempos aleatorios para las puntas (solo periodo interictal)
    spike_times = np.random.uniform(1, sz_onset - 1, n_spikes)
    
    for t_spike in spike_times:
        # Amplitud de la punta (100-300 uV, mucho mayor que el fondo)
        amp = np.random.uniform(100, 300)
        
        # Ancho de la punta (sigma): 5-15 ms
        # sigma controla la duracion: punta dura ~4*sigma
        sigma = np.random.uniform(0.005, 0.015)  # en segundos
        
        # Funcion Gaussiana centrada en t_spike
        # exp(-(t-t0)^2 / (2*sigma^2)) es la campana de Gauss
        gaussian = np.exp(-((t - t_spike)**2) / (2 * sigma**2))
        
        # Polaridad aleatoria (las puntas pueden ser positivas o negativas)
        polarity = np.random.choice([-1, 1])
        
        biomarker_signal += polarity * amp * gaussian
    
    # --------------------------------------------------------
    # HFOs (Ripples: 80-250 Hz)
    # --------------------------------------------------------
    # Los HFOs son bursts cortos de oscilaciones de alta frecuencia.
    # Matematicamente: sinusoide * envolvente gaussiana
    
    hfo_rate = 0.8  # HFOs por segundo
    n_hfos = int(hfo_rate * interictal_duration)
    hfo_times = np.random.uniform(1, sz_onset - 1, n_hfos)
    
    for t_hfo in hfo_times:
        # Frecuencia del HFO (ripple: 80-250 Hz)
        f_hfo = np.random.uniform(80, 250)
        
        # Amplitud (5-20 uV - mucho menor que las puntas)
        amp_hfo = np.random.uniform(5, 20)
        
        # Duracion del burst: 50-150 ms
        sigma_hfo = np.random.uniform(0.02, 0.06)
        
        # HFO = sinusoide de alta frecuencia * envolvente gaussiana
        envelope = np.exp(-((t - t_hfo)**2) / (2 * sigma_hfo**2))
        oscillation = np.sin(2 * np.pi * f_hfo * t)
        
        biomarker_signal += amp_hfo * oscillation * envelope
    
    return biomarker_signal

print("Funcion generate_interictal_biomarkers() definida correctamente.")
print(f"Genera ~{1.5*30:.0f} puntas y ~{0.8*30:.0f} HFOs en el periodo interictal.")

## 1.7 Ensamblar la Senal SEEG Completa

Ahora combinamos los tres componentes para cada canal:

$$\text{SEEG}(t) = \text{Background}(t) + \text{Ictal}(t) + \text{Biomarkers}(t)$$

Esto es una **superposicion lineal** - exactamente el principio de combinacion
lineal que subyace al algebra lineal.

In [ ]:
# ============================================================
# ENSAMBLAR SENAL SEEG COMPLETA
# ============================================================
# La senal final es la SUMA (superposicion lineal) de:
#   1. Actividad de fondo (siempre presente)
#   2. Actividad ictal (solo durante la crisis)
#   3. Biomarcadores interictales (solo en EZ, solo periodo interictal)

# Reiniciar semilla para reproducibilidad
np.random.seed(42)

# Matriz de senales: filas = canales, columnas = muestras
# Esta es la REPRESENTACION MATRICIAL de las senales multicanal
# (referencia: Eje 1 - la senal multicanal es una MATRIZ)
seeg_data = np.zeros((n_channels, n_samples))

print("Generando senales SEEG canal por canal...")
print("="*50)

for ch_idx in range(n_channels):
    zone = zone_labels[ch_idx]
    name = channel_names[ch_idx]
    
    # 1. Actividad de fondo (siempre presente)
    background = generate_background(n_samples, fs, zone)
    
    # 2. Actividad ictal (solo durante la crisis, solo EZ y PZ)
    ictal = generate_ictal_activity(n_samples, fs, zone,
                                    seizure_onset_ez, seizure_onset_pz, seizure_end)
    
    # 3. Biomarcadores interictales (solo en EZ)
    biomarkers = generate_interictal_biomarkers(n_samples, fs, zone, seizure_onset_ez)
    
    # SUPERPOSICION LINEAL: la senal final es la suma
    seeg_data[ch_idx, :] = background + ictal + biomarkers
    
    # Estadisticas del canal
    std_val = np.std(seeg_data[ch_idx, :])
    print(f"  {name:15s} | zona={zone:3s} | std={std_val:6.1f} uV")

print("="*50)
print(f"\nSenal SEEG generada exitosamente!")
print(f"Forma de la matriz: {seeg_data.shape}")
print(f"  -> {seeg_data.shape[0]} canales x {seeg_data.shape[1]} muestras")
print(f"  -> Cada fila es un canal, cada columna es un instante de tiempo")

## 1.8 Visualizacion: Trazas SEEG (Vista Clinica)

Esta es la visualizacion clasica que un epileptologo ve en la sala de monitoreo:
multiples canales apilados verticalmente, con el tiempo en el eje horizontal.

Busque visualmente:
- Periodo interictal (0-30s): actividad de fondo + puntas en EZ
- Inicio de crisis (~30s): cambio abrupto en canales EZ
- Propagacion (~33s): cambio en canales PZ
- Canales NIZ: sin cambios significativos

In [ ]:
# ============================================================
# VISUALIZACION 1: Registro SEEG completo (60 segundos)
# ============================================================
# Estilo 'waterfall': canales apilados verticalmente
# Cada canal se desplaza verticalmente para evitar superposicion

fig, ax = plt.subplots(figsize=(16, 12))

# Desplazamiento vertical entre canales
spacing = 400  # uV entre canales

# Colores por zona
zone_colors = {'EZ': 'red', 'PZ': 'orange', 'NIZ': 'steelblue'}

for ch_idx in range(n_channels):
    zone = zone_labels[ch_idx]
    offset = ch_idx * spacing
    
    ax.plot(t, seeg_data[ch_idx, :] + offset,
            color=zone_colors[zone],
            linewidth=0.4,
            alpha=0.8)

# Marcar los periodos
ax.axvline(x=seizure_onset_ez, color='red', linestyle='--', linewidth=2,
           label=f'Inicio crisis EZ ({seizure_onset_ez}s)')
ax.axvline(x=seizure_onset_pz, color='orange', linestyle='--', linewidth=2,
           label=f'Propagacion PZ ({seizure_onset_pz}s)')
ax.axvline(x=seizure_end, color='gray', linestyle='--', linewidth=2,
           label=f'Fin crisis ({seizure_end}s)')

# Sombrear el periodo ictal
ax.axvspan(seizure_onset_ez, seizure_end, alpha=0.1, color='red',
           label='Periodo ictal')

# Etiquetas de canales
ax.set_yticks([i * spacing for i in range(n_channels)])
ax.set_yticklabels(channel_names)

ax.set_xlabel('Tiempo (s)', fontsize=12)
ax.set_title('Registro SEEG Simulado - 12 Canales (60 segundos)',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, duration)

plt.tight_layout()
plt.show()

print("\nLeyenda de colores:")
print("  ROJO = Zona Epileptogenica (EZ) - Origen de la crisis")
print("  NARANJA = Zona de Propagacion (PZ) - Reclutada con retraso")
print("  AZUL = Zona No Involucrada (NIZ) - Actividad normal")

## 1.9 Visualizacion: Detalle del Inicio de Crisis

Ampliemos la ventana alrededor del inicio de la crisis para ver:
- La **LVFA** (actividad rapida de bajo voltaje) en los canales EZ
- El **retraso** de 3 segundos antes de que la PZ se involucre
- Los canales NIZ que NO cambian

In [ ]:
# ============================================================
# VISUALIZACION 2: Detalle del inicio ictal (25-40 segundos)
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Ventana temporal de interes
t_start, t_end = 25, 40  # segundos
mask = (t >= t_start) & (t <= t_end)

# --- Panel 1: Canales EZ ---
ax = axes[0]
ax.set_title('Zona Epileptogenica (EZ) - Hipocampo', fontsize=12, fontweight='bold', color='red')
for ch_idx in range(3):  # Primeros 3 canales = EZ
    ax.plot(t[mask], seeg_data[ch_idx, mask] + ch_idx*300,
            color='red', linewidth=0.6, label=channel_names[ch_idx])
ax.axvline(x=seizure_onset_ez, color='red', linestyle='--', linewidth=2)
ax.set_ylabel('Amplitud (uV)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.2)

# --- Panel 2: Canales PZ ---
ax = axes[1]
ax.set_title('Zona de Propagacion (PZ) - Amigdala/Insula', fontsize=12, fontweight='bold', color='orange')
for ch_idx in range(3, 6):  # Canales 3-5 = PZ
    ax.plot(t[mask], seeg_data[ch_idx, mask] + (ch_idx-3)*300,
            color='orange', linewidth=0.6, label=channel_names[ch_idx])
ax.axvline(x=seizure_onset_pz, color='orange', linestyle='--', linewidth=2)
ax.set_ylabel('Amplitud (uV)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.2)

# --- Panel 3: Canales NIZ ---
ax = axes[2]
ax.set_title('Zona No Involucrada (NIZ) - Temporal/Frontal', fontsize=12, fontweight='bold', color='steelblue')
for ch_idx in range(6, 9):  # 3 canales NIZ como ejemplo
    ax.plot(t[mask], seeg_data[ch_idx, mask] + (ch_idx-6)*300,
            color='steelblue', linewidth=0.6, label=channel_names[ch_idx])
ax.set_ylabel('Amplitud (uV)')
ax.set_xlabel('Tiempo (s)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.2)

# Marcar periodos en todos los paneles
for ax in axes:
    ax.axvspan(seizure_onset_ez, t_end, alpha=0.08, color='red')

plt.suptitle('Detalle del Inicio de Crisis (25-40s)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Observe:")
print("  1. La EZ muestra cambio abrupto a los 30s (LVFA)")
print("  2. La PZ muestra cambio a los 33s (3s de retraso)")
print("  3. La NIZ NO muestra cambios significativos")
print("  4. Este retraso es clave para el Indice de Epileptogenicidad (EI)")

## 1.10 Analisis Espectral Basico: Densidad Espectral de Potencia

La **Densidad Espectral de Potencia (PSD)** nos dice cuanta energia tiene la senal
en cada frecuencia. Se calcula usando el **metodo de Welch**:

1. Se divide la senal en segmentos superpuestos
2. Se calcula la FFT (Transformada Rapida de Fourier) de cada segmento
3. Se promedia el cuadrado de la magnitud

Esto nos permite verificar que nuestra simulacion tiene las propiedades espectrales
esperadas para cada zona.

In [ ]:
# ============================================================
# ANALISIS ESPECTRAL: PSD por zona (periodo interictal)
# ============================================================
# Calculamos la PSD solo del periodo INTERICTAL (0-30s)
# para ver las diferencias basales entre zonas

# Extraer solo el periodo interictal
interictal_end_idx = int(seizure_onset_ez * fs)
interictal_data = seeg_data[:, :interictal_end_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: PSD completa (0-100 Hz) ---
ax = axes[0]

for zone_name, color in [('EZ', 'red'), ('PZ', 'orange'), ('NIZ', 'steelblue')]:
    # Seleccionar canales de esta zona
    zone_channels = [i for i, z in enumerate(zone_labels) if z == zone_name]
    
    # Calcular PSD promedio de la zona
    # welch() implementa el metodo de Welch para estimacion espectral
    # nperseg = longitud de cada segmento (2 segundos * fs)
    psds = []
    for ch in zone_channels:
        freqs, psd = welch(interictal_data[ch, :], fs=fs, nperseg=2*fs)
        psds.append(psd)
    
    # Promedio y desviacion estandar entre canales de la misma zona
    psd_mean = np.mean(psds, axis=0)
    psd_std = np.std(psds, axis=0)
    
    # Graficar en escala logaritmica (dB)
    # 10*log10(PSD) convierte a decibeles
    ax.semilogy(freqs, psd_mean, color=color, linewidth=2, label=zone_name)
    ax.fill_between(freqs, psd_mean - psd_std, psd_mean + psd_std,
                    color=color, alpha=0.15)

ax.set_xlabel('Frecuencia (Hz)', fontsize=12)
ax.set_ylabel('PSD (uV^2/Hz)', fontsize=12)
ax.set_title('Densidad Espectral de Potencia - Interictal', fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Marcar bandas de frecuencia
bands = {'delta': (0.5,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,80)}
colors_bands = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd']
for (bname, (f1,f2)), bcol in zip(bands.items(), colors_bands):
    ax.axvspan(f1, f2, alpha=0.06, color=bcol)
    ax.text((f1+f2)/2, ax.get_ylim()[1]*0.5, bname, ha='center', fontsize=8, color=bcol)

# --- Panel 2: Potencia por banda (barras) ---
ax = axes[1]

# Calcular potencia por banda para cada zona
band_power = {zone: {} for zone in ['EZ', 'PZ', 'NIZ']}

for zone_name in ['EZ', 'PZ', 'NIZ']:
    zone_channels = [i for i, z in enumerate(zone_labels) if z == zone_name]
    for band_name, (f_low, f_high) in bands.items():
        powers = []
        for ch in zone_channels:
            freqs, psd = welch(interictal_data[ch, :], fs=fs, nperseg=2*fs)
            # Integrar la PSD en el rango de la banda
            # Esto da la potencia total en esa banda
            freq_mask = (freqs >= f_low) & (freqs <= f_high)
            band_pow = np.trapz(psd[freq_mask], freqs[freq_mask])
            powers.append(band_pow)
        band_power[zone_name][band_name] = np.mean(powers)

# Graficar barras agrupadas
x = np.arange(len(bands))
width = 0.25

for i, (zone_name, color) in enumerate([('EZ', 'red'), ('PZ', 'orange'), ('NIZ', 'steelblue')]):
    values = [band_power[zone_name][b] for b in bands.keys()]
    ax.bar(x + i*width, values, width, label=zone_name, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels([b.capitalize() for b in bands.keys()])
ax.set_ylabel('Potencia (uV^2)', fontsize=12)
ax.set_title('Potencia por Banda - Interictal', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nObservaciones esperadas:")
print("  - EZ: Mayor potencia en theta (hiperexcitabilidad) y delta")
print("  - NIZ: Mayor potencia en alpha (ritmo de fondo normal)")
print("  - Todas las zonas: espectro tipo 1/f (potencia decae con la frecuencia)")

## 1.11 La Senal SEEG como Matriz

Un concepto fundamental del **Eje 1** del examen: las senales multicanal
son una **MATRIZ** donde:
- Cada **fila** es un canal (un punto de registro en el cerebro)
- Cada **columna** es un instante de tiempo
- El **valor** en posicion (i, j) es el voltaje del canal i en el tiempo j

Esta representacion matricial es la base de TODO el analisis posterior:
- Conectividad funcional = relaciones entre FILAS
- Grafos cerebrales = grafos cuyos NODOS son filas de esta matriz
- PCA/ICA = transformaciones lineales de esta matriz

In [ ]:
# ============================================================
# LA SENAL SEEG COMO OBJETO MATRICIAL
# ============================================================

print("La senal SEEG es una MATRIZ:")
print(f"  Forma: {seeg_data.shape}")
print(f"  = {n_channels} canales x {n_samples} muestras")
print(f"")
print(f"Primeras 8 muestras (8/{fs} = {8/fs*1000:.1f} ms) de cada canal:")
print(f"{'Canal':15s} | Muestras...")
print("-"*70)
for ch_idx in range(n_channels):
    values = seeg_data[ch_idx, :8]
    vals_str = '  '.join([f'{v:7.1f}' for v in values])
    print(f"{channel_names[ch_idx]:15s} | {vals_str}")

print(f"\n" + "="*70)
print(f"Estadisticas por zona:")
print(f"{'Zona':5s} | {'Media':>10s} | {'Std':>10s} | {'Min':>10s} | {'Max':>10s}")
print("-"*55)
for zone_name in ['EZ', 'PZ', 'NIZ']:
    zone_channels = [i for i, z in enumerate(zone_labels) if z == zone_name]
    zone_data = seeg_data[zone_channels, :]
    print(f"{zone_name:5s} | {np.mean(zone_data):10.2f} | {np.std(zone_data):10.2f} | "
          f"{np.min(zone_data):10.2f} | {np.max(zone_data):10.2f}")

print(f"\nNota: La EZ tiene mayor variabilidad (std) debido a")
print(f"las puntas intercriticas y la actividad ictal intensa.")

## 1.12 Guardar Datos para los Pasos Siguientes

Guardamos la senal simulada y los metadatos para usarlos en los siguientes pasos
del analisis (conectividad, grafos, HFOs, GNN).

In [ ]:
# ============================================================
# GUARDAR DATOS PARA PASOS SIGUIENTES
# ============================================================

# Guardar en un diccionario de numpy
np.savez('../12 - Simulacion de Senales EEG/seeg_simulado.npz',
         seeg_data=seeg_data,             # Matriz de senales (12 x 61440)
         t=t,                              # Vector de tiempo
         fs=np.array(fs),                  # Frecuencia de muestreo
         channel_names=channel_names,       # Nombres de canales
         zone_labels=zone_labels,           # Etiquetas de zona
         seizure_onset_ez=np.array(seizure_onset_ez),
         seizure_onset_pz=np.array(seizure_onset_pz),
         seizure_end=np.array(seizure_end))

print("Datos guardados en 'seeg_simulado.npz'")
print("\nContenido del archivo:")
print("  seeg_data      : Matriz de senales SEEG")
print("  t              : Vector de tiempo")
print("  fs             : Frecuencia de muestreo")
print("  channel_names  : Nombres de canales")
print("  zone_labels    : Etiquetas de zona (EZ/PZ/NIZ)")
print("  seizure_onset_* : Tiempos de la crisis")

## Resumen del Paso 1

### Que hicimos:
1. Simulamos una senal SEEG de **12 canales** con tres zonas (EZ, PZ, NIZ)
2. Incluimos actividad de fondo diferenciada por zona
3. Simulamos una crisis con **LVFA** en EZ y **propagacion retardada** a PZ
4. Agregamos **puntas intercriticas** y **HFOs** en la EZ
5. Verificamos el contenido espectral con PSD

### Conceptos de algebra lineal utilizados:
- **Combinacion lineal**: la senal es suma de sinusoides (vectores base)
- **Representacion matricial**: senales multicanal = matriz
- **Superposicion**: background + ictal + biomarkers

### Conceptos clinicos cubiertos (del examen):
- Zonas EZ, PZ, NIZ (Ejes 1-4)
- LVFA como patron de inicio ictal (base del EI, Eje 2)
- Puntas y HFOs como biomarcadores interictales (Eje 3)
- Retraso de propagacion (clave para EI y conectividad dirigida)

### Siguiente paso:
**Step 2**: Analisis tiempo-frecuencia e Indice de Epileptogenicidad (EI)
- Espectrogramas para visualizar la evolucion temporal
- Calculo del EI segun Bartolomei et al. (2008)
- Discriminacion EZ vs PZ vs NIZ